# Strong-Strong Tracking Task

This notebook builds the same strong-strong tracking case as `examples/strong_strong_tracking.jl`, but all runtime choices are ordinary Julia variables. It uses no shell environment-variable overrides.

The configuration cell is set up for a long CUDA PIC run by default. All user-adjustable values are defined there once. For a quick smoke test, reduce `input.total_turns`, `input.electron.n_macro`, and `input.proton.n_macro`, or set `input.use_gpu = false`.

PIC luminosity is deposited at each slice pair's common centroid plane. By default, `pic_luminosity_deposit_method = nothing` inherits the force deposition method, and `pic_luminosity_grid = nothing` inherits the force-grid dimensions while retaining an independent luminosity workspace. With `pic_luminosity_every = 100`, the luminosity file contains only evaluated turns (`0, 100, 200, ...`); skipped turns are not written.


In [ ]:
import Pkg
project_root = abspath(joinpath(@__DIR__, ".."))
Pkg.activate(project_root)

if !isdefined(Main, :Octopus)
    include(joinpath(project_root, "src", "Octopus.jl"))
end
using .Octopus

println("Project root: ", project_root)

In [ ]:
# Complete user configuration. Edit controllable values only in this cell.
crossing_angle = 12.5e-3
electron_beta = (0.55, 0.056, 0.7e-2 / 5.5e-4)
electron_crab_beta = (150.0, 30.0, electron_beta[3])
proton_beta = (0.8, 0.072, 6.0e-2 / 6.6e-4)
proton_crab_beta = (1300.0, 30.0, proton_beta[3])

input = (
    case_name = "pic_hcc_notebook",
    result_dir = joinpath(project_root, "result"),
    seed = 123456789,
    total_turns = 50_000,
    use_gpu = true,
    cuda_device = nothing,          # keep current device; use 0, 1, ... to select
    solver_kind = :PIC,             # :PIC or :gaussian
    crossing_angle = crossing_angle,

    electron = (
        charge = -1.0, mass = EMASS_EV, energy = 10.0e9,
        n_particle = 1.7203e11, n_macro = 2_560_000, cutoff = 5.0, rng_id = 1,
        sigma = (106.0e-6, 9.5e-6, 0.7e-2), beta = electron_beta,
        alpha = (0.0, 0.0, 0.0),
        crab_beta = electron_crab_beta, tune = (0.08, 0.14, -0.069),
        chromaticity = (1.0, 1.0), crab_frequency = 394.0e6,
        crab_strength_x = (tan(crossing_angle) / sqrt(electron_crab_beta[1] * electron_beta[1]), 0.0, 0.0),
        crab_strength_y = (0.0, 0.0, 0.0), crab_phase = (0.0, 0.0, 0.0),
        radiation = (
            damping_turns = (4000.0, 4000.0, 2000.0), rng_id = 3,
            is_damping = true, is_excitation = true,
        ),
    ),

    proton = (
        charge = 1.0, mass = PMASS_EV, energy = 275.0e9,
        n_particle = 0.6881e11, n_macro = 1_024_000, cutoff = 5.0, rng_id = 2,
        sigma = (95.0e-6, 8.5e-6, 6.0e-2), beta = proton_beta,
        alpha = (0.0, 0.0, 0.0),
        crab_beta = proton_crab_beta, tune = (0.228, 0.210, -0.01),
        chromaticity = (2.0, 2.0), crab_frequency = 197.0e6,
        crab_strength_x = (
            tan(crossing_angle) / sqrt(proton_crab_beta[1] * proton_beta[1]) * 4.0 / 3.0,
            -tan(crossing_angle) / sqrt(proton_crab_beta[1] * proton_beta[1]) / 3.0,
            0.0,
        ),
        crab_strength_y = (0.0, 0.0, 0.0), crab_phase = (0.0, 0.0, 0.0),
    ),

    transport = (crab_to_ip_dmu = (pi / 2.0, 0.0, 0.0),),
    slicing = (method = :normal_quantile, zslice = 15, center = :centroid),
    solver = (
        pic_grid = (128, 128), pic_deposit_method = :CIC,
        pic_green_type = :integrated, pic_green_cache = :slice_pair,
        pic_slice_pair_green_min_ratio = 0.50,
        pic_slice_pair_green_growth = 0.25,
        pic_longitudinal_kick = true, pic_batch_mode = :wavefront,
        pic_luminosity_every = 100,  # evaluates turns 0, 100, 200, ...; 0 disables
        pic_luminosity_grid = nothing, # inherit pic_grid dimensions; workspace remains separate
        pic_luminosity_deposit_method = nothing, # inherit pic_deposit_method (:CIC or :TSC)
        cuda_pic_async = true, cuda_pic_batch_fft = true,
        cuda_pic_wavefront_fft = true, cuda_pic_indexed_wavefront = true,
        min_sigma = 1.0e-12, luminosity_scale = nothing,
    ),
    diagnostics = (
        record_turn_times = false, memory_log_every = 0, cache_stats = false,
        pic_timing = false, pic_timing_detail = false, nvtx = false,
    ),
    output = (
        luminosity_suffix = ".lum", electron_moment_suffix = ".ele.h5",
        proton_moment_suffix = ".pro.h5",
        moment_start = 0, moment_step = 1, moment_capacity = 100,
    ),
)

if input.use_gpu
    import CUDA
    CUDA.functional(false) || error("use_gpu=true requested, but CUDA.functional(false) is false.")
    policy = GPUExecutionPolicy(device = input.cuda_device)
else
    policy = CPUThreadsExecutionPolicy()
end

input


In [ ]:
set_global_rng!(seed = input.seed, method = :philox)
input.case_name


In [ ]:
ele = input.electron
beam_ele = Beam(ele.n_macro, policy, Float64;
    beta = ele.beta,
    alpha = ele.alpha,
    sigma = ele.sigma,
    cutoff = ele.cutoff,
    rng_id = ele.rng_id,
    charge = ele.charge,
    mc2 = ele.mass,
    E0 = ele.energy,
    r0 = RE * ME0 / ele.mass,
    npart = ele.n_particle,
)

pro = input.proton
beam_pro = Beam(pro.n_macro, policy, Float64;
    beta = pro.beta,
    alpha = pro.alpha,
    sigma = pro.sigma,
    cutoff = pro.cutoff,
    rng_id = pro.rng_id,
    charge = pro.charge,
    mc2 = pro.mass,
    E0 = pro.energy,
    r0 = RE * ME0 / pro.mass,
    npart = pro.n_particle,
)

eltype(beam_ele.rep.x) === Float64 || error("electron beam tracking arrays must be Float64")
eltype(beam_pro.rep.x) === Float64 || error("proton beam tracking arrays must be Float64")

(beam_statistics(beam_ele).rms, beam_statistics(beam_pro).rms)

In [ ]:
solver_input = input.solver
slicing = LongitudinalSlicing(;
    method = input.slicing.method,
    nslices = input.slicing.zslice,
    center_position = input.slicing.center,
)

pic_luminosity_every = solver_input.pic_luminosity_every
pic_luminosity_schedule =
    pic_luminosity_every < 0 ? error("pic_luminosity_every must be >= 0") :
    pic_luminosity_every == 0 ? AtTurns(Int[]) :
    pic_luminosity_every == 1 ? nothing :
    EveryNSteps(step = pic_luminosity_every)

solver = if input.solver_kind == :gaussian
    GaussianPoissonSolver(;
        slicing = slicing,
        min_sigma = solver_input.min_sigma,
        luminosity_scale = solver_input.luminosity_scale,
    )
elseif input.solver_kind == :PIC
    PICPoissonSolver(;
        slicing = slicing,
        luminosity_scale = solver_input.luminosity_scale,
        grid = solver_input.pic_grid,
        deposit_method = solver_input.pic_deposit_method,
        green_type = solver_input.pic_green_type,
        green_cache = solver_input.pic_green_cache,
        slice_pair_green_min_ratio = solver_input.pic_slice_pair_green_min_ratio,
        slice_pair_green_growth = solver_input.pic_slice_pair_green_growth,
        longitudinal_kick = solver_input.pic_longitudinal_kick,
        batch_mode = solver_input.pic_batch_mode,
        cuda_async = solver_input.cuda_pic_async,
        cuda_batch_fft = solver_input.cuda_pic_batch_fft,
        cuda_wavefront_fft = solver_input.cuda_pic_wavefront_fft,
        cuda_indexed_wavefront = solver_input.cuda_pic_indexed_wavefront,
        luminosity_grid = solver_input.pic_luminosity_grid,
        luminosity_deposit_method = solver_input.pic_luminosity_deposit_method,
        luminosity_schedule = pic_luminosity_schedule,
    )
else
    error("unknown solver_kind=$(input.solver_kind); use :PIC or :gaussian")
end

solver_config = solver_configuration(solver)
solver_config


In [ ]:
electron_tccb2ip = Linear6DSpec{Float64}(;
    beta1 = ele.crab_beta,
    beta2 = ele.beta,
    alpha1 = ele.alpha,
    alpha2 = ele.alpha,
    dmu = input.transport.crab_to_ip_dmu,
)
electron_tccb2ip_inv = Linear6DSpec{Float64}(matrix = inv(Matrix(Linear6D(electron_tccb2ip))))

electron_ip2tcca = Linear6DSpec{Float64}(;
    beta1 = ele.beta,
    beta2 = ele.crab_beta,
    alpha1 = ele.alpha,
    alpha2 = ele.alpha,
    dmu = input.transport.crab_to_ip_dmu,
)
electron_ip2tcca_inv = Linear6DSpec{Float64}(matrix = inv(Matrix(Linear6D(electron_ip2tcca))))

electron_tccb = ThinCrabCavitySpec{3}(ele.crab_frequency;
    strengthX = ele.crab_strength_x,
    strengthY = ele.crab_strength_y,
    phase = ele.crab_phase,
)
electron_tcca = ThinCrabCavitySpec{3}(ele.crab_frequency;
    strengthX = ele.crab_strength_x,
    strengthY = ele.crab_strength_y,
    phase = ele.crab_phase,
)

electron_one_turn = Linear6DSpec{Float64}(;
    beta1 = ele.beta,
    beta2 = ele.beta,
    alpha1 = ele.alpha,
    alpha2 = ele.alpha,
    dmu = 2pi .* ele.tune,
)
electron_chrom = ChromaticityKickSpec{Float64}(;
    xi = ele.chromaticity,
    beta = ele.beta,
    alpha = ele.alpha,
)
electron_rad = LumpedRadSpec{Float64}(;
    damping_turns = ele.radiation.damping_turns,
    beta = ele.beta,
    alpha = ele.alpha,
    sigma = ele.sigma,
    is_damping = ele.radiation.is_damping,
    is_excitation = ele.radiation.is_excitation,
    rng_id = ele.radiation.rng_id,
)

proton_tccb2ip = Linear6DSpec{Float64}(;
    beta1 = pro.crab_beta,
    beta2 = pro.beta,
    alpha1 = pro.alpha,
    alpha2 = pro.alpha,
    dmu = input.transport.crab_to_ip_dmu,
)
proton_tccb2ip_inv = Linear6DSpec{Float64}(matrix = inv(Matrix(Linear6D(proton_tccb2ip))))

proton_ip2tcca = Linear6DSpec{Float64}(;
    beta1 = pro.beta,
    beta2 = pro.crab_beta,
    alpha1 = pro.alpha,
    alpha2 = pro.alpha,
    dmu = input.transport.crab_to_ip_dmu,
)
proton_ip2tcca_inv = Linear6DSpec{Float64}(matrix = inv(Matrix(Linear6D(proton_ip2tcca))))

proton_tccb = ThinCrabCavitySpec{3}(pro.crab_frequency;
    strengthX = pro.crab_strength_x,
    strengthY = pro.crab_strength_y,
    phase = pro.crab_phase,
)
proton_tcca = ThinCrabCavitySpec{3}(pro.crab_frequency;
    strengthX = pro.crab_strength_x,
    strengthY = pro.crab_strength_y,
    phase = pro.crab_phase,
)

proton_one_turn = Linear6DSpec{Float64}(;
    beta1 = pro.beta,
    beta2 = pro.beta,
    alpha1 = pro.alpha,
    alpha2 = pro.alpha,
    dmu = 2pi .* pro.tune,
)
proton_chrom = ChromaticityKickSpec{Float64}(;
    xi = pro.chromaticity,
    beta = pro.beta,
    alpha = pro.alpha,
)

lb = LorentzBoostSpec(input.crossing_angle)
rlb = RevLorentzBoostSpec(input.crossing_angle)
ip = StrongStrongCollision(:ip; poisson_solver = solver)

"elements ready"

In [ ]:
mkpath(input.result_dir)
luminosity_path = joinpath(input.result_dir, input.case_name * input.output.luminosity_suffix)
electron_moment_path = joinpath(input.result_dir, input.case_name * input.output.electron_moment_suffix)
proton_moment_path = joinpath(input.result_dir, input.case_name * input.output.proton_moment_suffix)
moment_schedule = EveryNSteps(;
    start = input.output.moment_start,
    stop = input.total_turns,
    step = input.output.moment_step,
)

line_ele = (
    electron_tccb2ip_inv,
    electron_tccb,
    electron_tccb2ip,
    lb,
    ip,
    rlb,
    electron_ip2tcca,
    electron_tcca,
    electron_ip2tcca_inv,
    electron_one_turn,
    electron_chrom,
    electron_rad,
    ScheduledObserver(
        MomentObserver(electron_moment_path; capacity = input.output.moment_capacity),
        moment_schedule,
    ),
)

line_pro = (
    proton_tccb2ip_inv,
    proton_tccb,
    proton_tccb2ip,
    lb,
    ip,
    rlb,
    proton_ip2tcca,
    proton_tcca,
    proton_ip2tcca_inv,
    proton_one_turn,
    proton_chrom,
    ScheduledObserver(
        MomentObserver(proton_moment_path; capacity = input.output.moment_capacity),
        moment_schedule,
    ),
)

diagnostics = StrongStrongDiagnostics(; input.diagnostics...)
task = StrongStrongTask(line_ele, line_pro;
    luminosity_path = luminosity_path,
    diagnostics = diagnostics,
)

(length(line_ele), length(line_pro), luminosity_path, electron_moment_path, proton_moment_path)

In [ ]:
@time execute!(task, beam_ele, beam_pro; turns = input.total_turns)

stats_ele = beam_statistics(beam_ele)
stats_pro = beam_statistics(beam_pro)
println("turns = ", input.total_turns)
println("n_macro_ele = ", ele.n_macro)
println("n_macro_pro = ", pro.n_macro)
println("use_gpu = ", input.use_gpu)
println("solver_kind = ", input.solver_kind)
if input.solver_kind == :PIC
    println("pic_longitudinal_kick = ", solver_input.pic_longitudinal_kick)
    println("pic_batch_mode = ", solver_input.pic_batch_mode)
    println("pic_green_cache = ", solver_input.pic_green_cache)
    println("pic_slice_pair_green_min_ratio = ", solver_input.pic_slice_pair_green_min_ratio)
    println("pic_slice_pair_green_growth = ", solver_input.pic_slice_pair_green_growth)
    println("pic_luminosity_every = ", pic_luminosity_every)
    println("cuda_pic_async = ", solver_input.cuda_pic_async)
    println("cuda_pic_batch_fft = ", solver_input.cuda_pic_batch_fft)
    println("cuda_pic_wavefront_fft = ", solver_input.cuda_pic_wavefront_fft)
    println("cuda_pic_indexed_wavefront = ", solver_input.cuda_pic_indexed_wavefront)
    println("pic_luminosity_grid = ", solver_config.resolved_luminosity_grid)
    println("pic_luminosity_deposit_method = ",
            solver_input.pic_luminosity_deposit_method === nothing ? "inherit" : solver_input.pic_luminosity_deposit_method)
    println("pic_luminosity_deposit_method_resolved = ", solver_config.resolved_luminosity_deposit_method)
end
println("luminosity = ", luminosity_path)
println("electron moments = ", electron_moment_path)
println("proton moments = ", proton_moment_path)
println("electron rms = ", stats_ele.rms)
println("proton rms = ", stats_pro.rms)


In [ ]:
electron_output = MomentOutputFile(electron_moment_path)
proton_output = MomentOutputFile(proton_moment_path)

println("electron columns = ", column_names(electron_output))
println("proton columns = ", column_names(proton_output))
println("electron data size = ", size(read(electron_output)))
println("proton data size = ", size(read(proton_output)))